# Assignment 01

In [47]:
! pip install kaggle
import pandas as pd
import os


## Check for Missing Values

In [48]:
def check_missing_values(df: pd.DataFrame, df_name: str = "DataFrame"):

    if df.empty:
        print(f"\nWarning: {df_name} is empty. Cannot perform missing value check.")
        return

    
    separator_length = 60
    print("\n" + "="*separator_length)
   
    print(f"## Detailed Missing Value Check in {df_name}")
    print("="*separator_length)

   
    missing_values = df.isnull().sum()
    
    missing_values = missing_values[missing_values > 0]

    if missing_values.empty:
        print(f"No missing (NaN) values found in the {df_name}.")
    else:
        print(f"Missing values found for the following variables in {df_name}:")
        print("\nColumn Name | NaN Count")
        print("--------------------|----------")
        
        for column, count in missing_values.sort_values(ascending=False).items():
            print(f"{column.ljust(18)}| {count}")

    print("="*separator_length)

## Check for Duplicates

In [49]:
def check_for_duplicates(df: pd.DataFrame, df_name: str, key: list):
  
    separator_length = 60
    print("\n" + "="*separator_length)
    print(f"## Duplicate Entry Check for: {df_name}")
    print(f"Key used: {key}")
    print("="*separator_length)

    if df.empty:
        print(f"Warning: {df_name} is empty. Skipping duplicate check.")
        return

    duplicate_rows = df[df.duplicated(subset=key, keep=False)]

    if duplicate_rows.empty:
        print(f"No duplicate entries found for the key {tuple(key)} in {df_name}.")
    else:
        num_duplicates = len(duplicate_rows)
       
        num_groups = num_duplicates - duplicate_rows.drop_duplicates(subset=key).shape[0]
        
        print(f"{num_duplicates} duplicate rows identified based on the key {tuple(key)}.")
        print(f"(These rows belong to {num_groups} unique sets of duplicate key values.)")
        
        print("\nSample of Duplicate Rows (showing all entries for the duplicated key):")
      
        print(duplicate_rows.sort_values(by=key).head())
        
    print("="*separator_length)

## Dataset: Food Demand Forecasting

| Criterion | Details |
|------------|----------|
| **Name of the Dataset** | Food Demand Forecasting Dataset |
| **Source (Link)** | [Kaggle: Food Demand Forecasting](https://www.kaggle.com/competitions/food-demand-forecasting) |
| **Description of Data Measures** | The data measures the weekly number of orders (demand) for specific center-meal combinations within a multi-city meal delivery company. The goal is to predict future demand to optimize the planning of perishable raw material stock and staffing at various fulfillment centers. |
| **Frequency of the Data** | Weekly. The raw material replenishment and forecasting task are structured around weekly intervals. |
| **Time Span Covered** | The available historical data (training set) covers an unspecified period, ending at Week 145. The competition task requires forecasting demand for the subsequent 10 weeks (Weeks 146–155). |


In [50]:
! kaggle datasets download -d kannanaikkal/food-demand-forecasting -p df_food_demand --unzip

Dataset URL: https://www.kaggle.com/datasets/kannanaikkal/food-demand-forecasting
License(s): DbCL-1.0
  0%|                                               | 0.00/5.80M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 5.80M/5.80M [00:00<00:00, 1.49GB/s]


### Load dataset + merge

In [51]:
data_dir = 'df_food_demand' 

try:
    train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
    test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
    center_info_df = pd.read_csv(os.path.join(data_dir, 'fulfilment_center_info.csv'))
    meal_info_df = pd.read_csv(os.path.join(data_dir, 'meal_info.csv'))

    print("Dataframes loaded successfully!")
    print(f"Training data shape: {train_df.shape}")
    
   
    train_df = pd.merge(train_df, center_info_df, on='center_id', how='left')
    train_df = pd.merge(train_df, meal_info_df, on='meal_id', how='left')
    
    test_df = pd.merge(test_df, center_info_df, on='center_id', how='left')
    test_df = pd.merge(test_df, meal_info_df, on='meal_id', how='left')

    print("\nSample of Merged Training Data:")
    print(train_df.head())

except FileNotFoundError:
    print(f"\nError: Could not find files in the '{data_dir}' directory.")
    print("Please ensure the Kaggle API download command in Step 1 ran correctly.")

Dataframes loaded successfully!
Training data shape: (456548, 9)

Sample of Merged Training Data:
        id  week  center_id  meal_id  checkout_price  base_price  \
0  1379560     1         55     1885          136.83      152.29   
1  1466964     1         55     1993          136.83      135.83   
2  1346989     1         55     2539          134.86      135.86   
3  1338232     1         55     2139          339.50      437.53   
4  1448490     1         55     2631          243.50      242.50   

   emailer_for_promotion  homepage_featured  num_orders  city_code  \
0                      0                  0         177        647   
1                      0                  0         270        647   
2                      0                  0         189        647   
3                      0                  0          54        647   
4                      0                  0          40        647   

   region_code center_type  op_area   category cuisine  
0           56 

### Check for Missing Values

In [52]:
check_missing_values(train_df, df_name="Training Data")
check_missing_values(test_df, df_name="Test Data")


## Detailed Missing Value Check in Training Data
No missing (NaN) values found in the Training Data.

## Detailed Missing Value Check in Test Data
No missing (NaN) values found in the Test Data.


### Validate the Timestamp Sequence:

In [53]:
print("\n## Validate the Timestamp Sequence (Week Continuity)")


min_week = train_df['week'].min()
max_week = train_df['week'].max()
total_weeks = max_week - min_week + 1

print(f"Time Span Covered: Week {min_week} to Week {max_week}")
print(f"Total Unique Weeks Expected: {total_weeks}")

unique_weeks = train_df['week'].nunique()
print(f"Total Unique Weeks Found: {unique_weeks}")

if unique_weeks == total_weeks:
    print("All weeks (1 to 145) are present in the dataset (at least once).")
else:
    print(f"Warning: Missing week numbers in the overall time series. Expected {total_weeks}, found {unique_weeks}.")
    
# Check for internal series gaps (Are all weeks present for every center-meal combination?)
pivot_table = train_df.pivot_table(
    index=['center_id', 'meal_id'], 
    columns='week', 
    values='num_orders'
)

missing_weeks_per_series = pivot_table.isnull().sum(axis=1)
series_with_gaps = missing_weeks_per_series[missing_weeks_per_series > 0]

print(f"\nTotal unique center-meal combinations (time series): {len(pivot_table)}")
print(f"Number of series with missing weeks (internal gaps): {len(series_with_gaps)}")

if len(series_with_gaps) == 0:
    print("All unique center-meal combinations appear to have data for all weeks 1 to 145.")
else:
    print(f"Significant gaps found. {len(series_with_gaps)} out of {len(pivot_table)} series have missing weeks.")
    print("\nSample series with missing weeks (count of missing weeks):")
    print(series_with_gaps.head())
    print("\nThis means many (meal, center) combinations were not ordered for some weeks.")


## Validate the Timestamp Sequence (Week Continuity)
Time Span Covered: Week 1 to Week 145
Total Unique Weeks Expected: 145
Total Unique Weeks Found: 145
All weeks (1 to 145) are present in the dataset (at least once).

Total unique center-meal combinations (time series): 3597
Number of series with missing weeks (internal gaps): 2350
Significant gaps found. 2350 out of 3597 series have missing weeks.

Sample series with missing weeks (count of missing weeks):
center_id  meal_id
10         1207        1
           1216        2
           1247       23
           1438        2
           1525        8
dtype: int64

This means many (meal, center) combinations were not ordered for some weeks.


### Check for Duplicates

In [54]:
time_series_key = ['week', 'center_id', 'meal_id']


check_for_duplicates(train_df, "Merged Training Data", time_series_key)
check_for_duplicates(test_df, "Merged Test Data", time_series_key)


## Duplicate Entry Check for: Merged Training Data
Key used: ['week', 'center_id', 'meal_id']
No duplicate entries found for the key ('week', 'center_id', 'meal_id') in Merged Training Data.

## Duplicate Entry Check for: Merged Test Data
Key used: ['week', 'center_id', 'meal_id']
No duplicate entries found for the key ('week', 'center_id', 'meal_id') in Merged Test Data.
